In [ ]:
# ============================================================
# PHASE 4.5 — CELL 1: Environment Setup
# Runtime: CPU only. No GPU is needed anywhere in this notebook,
# since Phase 4.5 is data preparation, not model training or inference.
# ============================================================
#
# This notebook produces the corrected NTU RGB+D 120 CSub and CSet datasets.
# CSub is used for Phase 5 CTR-GCN benchmarking (verified result: Top-1
# 84.91 percent, matching the official published 84.9 percent). CSet is
# built with the identical method for completeness, though it has not yet
# been separately confirmed through an inference run.
#
# It reproduces every step from the original raw dataset zips through to
# the final normalized, correctly split .npy files, and documents the
# exact bugs found and fixed during development so the pipeline below
# does not need any further correction.

import os
import time
from google.colab import drive

# --- Mount Google Drive ---
drive.mount('/content/drive')
print("Drive mounted")

# --- Project paths on Drive ---
DRIVE_ROOT = '/content/drive/MyDrive/HRC_Research'
DRIVE_BACKUP = f'{DRIVE_ROOT}/datasets/NTU120/backup_important_files'
DRIVE_RAW_DATA = f'{DRIVE_BACKUP}/raw_data'
DRIVE_DENOISED_DATA = f'{DRIVE_BACKUP}/denoised_data'
DRIVE_CSUB_OUT = f'{DRIVE_BACKUP}/NTU120_CSub_v3'
DRIVE_CSET_OUT = f'{DRIVE_BACKUP}/NTU120_CSet_v3'

# --- Raw NTU skeleton zip files (source dataset, from ROSE Lab) ---
ZIP_NTU60 = '/content/drive/MyDrive/nturgbd_skeletons_s001_to_s017.zip'
ZIP_NTU120_EXT = '/content/drive/MyDrive/nturgbd_skeletons_s018_to_s032.zip'

os.makedirs(DRIVE_BACKUP, exist_ok=True)

# --- Clone a fresh CTR-GCN repo ---
# The official preprocessing scripts (get_raw_skes_data.py, get_raw_denoised_data.py,
# and the statistics files that define the correct train/test splits) live inside
# this repo. Cloning fresh avoids any leftover patches from a previous session.
if not os.path.exists('/content/CTR-GCN'):
    exit_code = os.system('git clone https://github.com/Uason-Chen/CTR-GCN.git /content/CTR-GCN')
    if exit_code != 0:
        raise RuntimeError("git clone failed. Check network access and repo URL.")
    print("CTR-GCN cloned")
else:
    print("CTR-GCN already present")

# --- Verify the statistics files we depend on later are present ---
# These define official subject IDs, camera setup IDs, and action labels for
# every one of the 113,945 valid NTU120 samples, in a fixed canonical order.
STAT_DIR = '/content/CTR-GCN/data/ntu120/statistics'
required_stat_files = ['performer.txt', 'setup.txt', 'label.txt', 'skes_available_name.txt']
print("\nChecking required statistics files:")
for fname in required_stat_files:
    fpath = os.path.join(STAT_DIR, fname)
    status = "OK" if os.path.exists(fpath) else "MISSING"
    print(f"  {fname}: {status}")
    if status == "MISSING":
        raise FileNotFoundError(f"Required statistics file missing: {fpath}")

print("\n=== Cell 1 complete. Environment ready. ===")


In [ ]:
# ============================================================
# PHASE 4.5 — CELL 2: Obtain raw_skes_data.pkl
# Runtime: CPU only.
#
# raw_skes_data.pkl is the output of the official get_raw_skes_data.py script.
# It contains every valid skeleton sequence read directly from the raw .skeleton
# files, before any denoising or normalization.
#
# This cell tries a FAST PATH first: if this file was already produced and backed
# up to Drive in an earlier session, it is simply copied over (a few minutes).
# If it is not present, this cell falls back to the SLOW PATH: extracting the raw
# NTU skeleton zip files and running the official script from scratch
# (expect roughly 1-2 hours for the full NTU RGB+D 120 skeleton set).
# ============================================================

import os
import shutil
import time
import re

LOCAL_NTU120_DIR = '/content/CTR-GCN/data/ntu120'
LOCAL_RAW_DATA = os.path.join(LOCAL_NTU120_DIR, 'raw_data')
LOCAL_RAW_PKL = os.path.join(LOCAL_RAW_DATA, 'raw_skes_data.pkl')
DRIVE_RAW_PKL = os.path.join(DRIVE_RAW_DATA, 'raw_skes_data.pkl')

os.makedirs(LOCAL_RAW_DATA, exist_ok=True)

if os.path.exists(DRIVE_RAW_PKL):
    # --- FAST PATH: reuse the already-verified backup from Drive ---
    print("Found existing raw_skes_data.pkl backup on Drive. Using fast path.")
    src_size = os.path.getsize(DRIVE_RAW_PKL)
    print(f"Copying raw_skes_data.pkl ({src_size/1024**3:.2f} GB) from Drive ...")
    t0 = time.time()
    shutil.copy2(DRIVE_RAW_PKL, LOCAL_RAW_PKL)
    dst_size = os.path.getsize(LOCAL_RAW_PKL)
    status = "OK" if dst_size == src_size else "SIZE MISMATCH"
    print(f"{status} ({(time.time()-t0)/60:.1f} min)")
    if status != "OK":
        raise RuntimeError("Copied file size does not match source. Re-run this cell.")

else:
    # --- SLOW PATH: extract from raw zips and run the official script ---
    print("No existing backup found on Drive. Running full raw extraction.")
    print("This step can take 1-2 hours. Do not interrupt the runtime.")

    LOCAL_SKELETONS = '/content/ntu120_skeletons'
    os.makedirs(LOCAL_SKELETONS, exist_ok=True)

    # Step 1: verify both source zip files exist
    for z in [ZIP_NTU60, ZIP_NTU120_EXT]:
        if not os.path.exists(z):
            raise FileNotFoundError(f"Required zip not found: {z}")
        size_gb = os.path.getsize(z) / 1024**3
        print(f"Found: {os.path.basename(z)} ({size_gb:.2f} GB)")

    # Step 2: copy zips to local disk (faster than extracting directly from Drive)
    local_zip_60 = '/content/nturgbd_s001_s017.zip'
    local_zip_120 = '/content/nturgbd_s018_s032.zip'
    for src, dst in [(ZIP_NTU60, local_zip_60), (ZIP_NTU120_EXT, local_zip_120)]:
        if not os.path.exists(dst):
            print(f"Copying {os.path.basename(src)} to local disk ...")
            t0 = time.time()
            shutil.copy2(src, dst)
            print(f"  Done in {(time.time()-t0)/60:.1f} min")

    # Step 3: extract both zips into the same folder.
    # NTU RGB+D 120 (full, 106 subjects) is the union of both zips' skeleton files.
    # Extracting both into one destination folder combines them automatically,
    # since the two zips contain non-overlapping sets of .skeleton files.
    for local_zip in [local_zip_60, local_zip_120]:
        print(f"\nExtracting {os.path.basename(local_zip)} ...")
        t0 = time.time()
        exit_code = os.system(f'unzip -q -o {local_zip} -d {LOCAL_SKELETONS}')
        print(f"  Done in {(time.time()-t0)/60:.1f} min, exit code: {exit_code}")
        if exit_code != 0:
            raise RuntimeError(f"unzip failed for {local_zip} with exit code {exit_code}")
        os.remove(local_zip)

    # Step 4: verify the extracted skeleton folder and file count.
    # Some zip layouts nest an extra "nturgb+d_skeletons" folder; handle both cases.
    candidate = os.path.join(LOCAL_SKELETONS, 'nturgb+d_skeletons')
    skeleton_dir = candidate if os.path.isdir(candidate) else LOCAL_SKELETONS

    from collections import Counter
    setups = Counter()
    for f in os.listdir(skeleton_dir):
        if f.endswith('.skeleton'):
            setups[f[:4]] += 1
    total_files = sum(setups.values())
    print(f"\nTotal .skeleton files found: {total_files}")
    if total_files == 0:
        raise RuntimeError("No .skeleton files found after extraction. Check zip contents.")

    # Step 5: patch and run the official get_raw_skes_data.py.
    # Patch 1: dtype=np.int is removed in modern NumPy, replace with dtype=int.
    # Patch 2: point skes_path at our actual extracted skeleton folder.
    script_path = os.path.join(LOCAL_NTU120_DIR, 'get_raw_skes_data.py')
    with open(script_path, 'r') as f:
        content = f.read()

    content = content.replace('dtype=np.int', 'dtype=int')
    content = re.sub(
        r"^(\s*)skes_path\s*=\s*'.*'",
        lambda m: f"{m.group(1)}skes_path = '{skeleton_dir}/'",
        content,
        flags=re.MULTILINE
    )
    with open(script_path, 'w') as f:
        f.write(content)
    print("\nPatched get_raw_skes_data.py")

    import subprocess
    print("Running get_raw_skes_data.py (this is the slow step) ...")
    t0 = time.time()
    result = subprocess.run(
        ['python', 'get_raw_skes_data.py'],
        cwd=LOCAL_NTU120_DIR,
        capture_output=True, text=True
    )
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"get_raw_skes_data.py failed with exit code {result.returncode}")
    print(f"Done in {(time.time()-t0)/60:.1f} min")

    if not os.path.exists(LOCAL_RAW_PKL):
        raise FileNotFoundError(f"Expected output not found: {LOCAL_RAW_PKL}")

# --- Verify the file we now have locally, regardless of which path was taken ---
size_gb = os.path.getsize(LOCAL_RAW_PKL) / 1024**3
print(f"\nraw_skes_data.pkl ready locally: {size_gb:.2f} GB")
print("\n=== Cell 2 complete. ===")


In [ ]:
# ============================================================
# PHASE 4.5 — CELL 3: Obtain raw_denoised_joints.pkl
# Runtime: CPU only.
#
# raw_denoised_joints.pkl is the output of the official get_raw_denoised_data.py
# script. It removes noisy or unreliable skeletons from raw_skes_data.pkl and
# selects the correct one or two main actors per sample based on motion amount.
#
# Same fast-path / slow-path pattern as Cell 2.
# ============================================================

import os
import shutil
import time
import subprocess

LOCAL_DENOISED_DIR = os.path.join(LOCAL_NTU120_DIR, 'denoised_data')
LOCAL_DENOISED_PKL = os.path.join(LOCAL_DENOISED_DIR, 'raw_denoised_joints.pkl')
LOCAL_FRAMES_CNT = os.path.join(LOCAL_DENOISED_DIR, 'frames_cnt.txt')
DRIVE_DENOISED_PKL = os.path.join(DRIVE_DENOISED_DATA, 'raw_denoised_joints.pkl')
DRIVE_FRAMES_CNT = os.path.join(DRIVE_DENOISED_DATA, 'frames_cnt.txt')

os.makedirs(LOCAL_DENOISED_DIR, exist_ok=True)

if os.path.exists(DRIVE_DENOISED_PKL) and os.path.exists(DRIVE_FRAMES_CNT):
    # --- FAST PATH: reuse the already-verified backup from Drive ---
    print("Found existing denoised_data backup on Drive. Using fast path.")
    for src, dst in [(DRIVE_DENOISED_PKL, LOCAL_DENOISED_PKL),
                      (DRIVE_FRAMES_CNT, LOCAL_FRAMES_CNT)]:
        src_size = os.path.getsize(src)
        print(f"Copying {os.path.basename(src)} ({src_size/1024**3:.3f} GB) ...", end=" ")
        t0 = time.time()
        shutil.copy2(src, dst)
        dst_size = os.path.getsize(dst)
        status = "OK" if dst_size == src_size else "SIZE MISMATCH"
        print(f"{status} ({(time.time()-t0)/60:.1f} min)")
        if status != "OK":
            raise RuntimeError(f"Copy failed for {src}. Re-run this cell.")

else:
    # --- SLOW PATH: run the official denoising script on raw_skes_data.pkl ---
    print("No existing backup found on Drive. Running get_raw_denoised_data.py.")

    script_path = os.path.join(LOCAL_NTU120_DIR, 'get_raw_denoised_data.py')
    with open(script_path, 'r') as f:
        content = f.read()
    content = content.replace('dtype=np.int', 'dtype=int')
    with open(script_path, 'w') as f:
        f.write(content)
    print("Patched get_raw_denoised_data.py")

    print("Running get_raw_denoised_data.py ...")
    t0 = time.time()
    result = subprocess.run(
        ['python', 'get_raw_denoised_data.py'],
        cwd=LOCAL_NTU120_DIR,
        capture_output=True, text=True
    )
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"get_raw_denoised_data.py failed with exit code {result.returncode}")
    print(f"Done in {(time.time()-t0)/60:.1f} min")

    if not os.path.exists(LOCAL_DENOISED_PKL):
        raise FileNotFoundError(f"Expected output not found: {LOCAL_DENOISED_PKL}")

size_gb = os.path.getsize(LOCAL_DENOISED_PKL) / 1024**3
print(f"\nraw_denoised_joints.pkl ready locally: {size_gb:.2f} GB")
print("\n=== Cell 3 complete. ===")


In [ ]:
# ============================================================
# PHASE 4.5 — CELL 4: Backup intermediate files to Drive
# Runtime: CPU only.
#
# raw_skes_data.pkl and raw_denoised_joints.pkl are expensive to regenerate
# (over an hour combined). Back them up to Drive now if they are not already
# there, so future sessions can use the fast path in Cells 2 and 3.
# ============================================================

import os
import shutil
import time

os.makedirs(DRIVE_RAW_DATA, exist_ok=True)
os.makedirs(DRIVE_DENOISED_DATA, exist_ok=True)

backup_pairs = [
    (LOCAL_RAW_PKL, os.path.join(DRIVE_RAW_DATA, 'raw_skes_data.pkl')),
    (LOCAL_DENOISED_PKL, os.path.join(DRIVE_DENOISED_DATA, 'raw_denoised_joints.pkl')),
    (LOCAL_FRAMES_CNT, os.path.join(DRIVE_DENOISED_DATA, 'frames_cnt.txt')),
]

for src, dst in backup_pairs:
    if os.path.exists(dst):
        print(f"Already backed up: {os.path.basename(dst)}")
        continue
    src_size = os.path.getsize(src)
    print(f"Backing up {os.path.basename(dst)} ({src_size/1024**3:.3f} GB) ...", end=" ")
    t0 = time.time()
    shutil.copy2(src, dst)
    dst_size = os.path.getsize(dst)
    status = "OK" if dst_size == src_size else "SIZE MISMATCH"
    print(f"{status} ({(time.time()-t0)/60:.1f} min)")

print("\n=== Cell 4 complete. Intermediate files safe on Drive. ===")


In [ ]:
# ============================================================
# PHASE 4.5 — CELL 5: Build the corrected NTU RGB+D 120 CSub and CSet datasets
# Runtime: CPU only.
#
# This is the step that was originally done incorrectly (see notes below),
# then fixed and verified. It reimplements the official seq_transformation.py
# logic exactly, because the official script itself cannot run on standard
# Colab (its align_frames step needs roughly 19+ GB of RAM for all 113,945
# NTU120 samples at once; Colab CPU sessions have about 12.7 GB).
#
# What the official logic actually does (confirmed by reading the real
# seq_transformation.py source directly, not from memory):
#
#   1. Centering (seq_translation): for each sample, find the first frame
#      where actor 1 has real (nonzero) data, take that frame spine-mid
#      joint (joint index 1, columns 3:6 of the 75-value per-actor vector)
#      as a fixed origin, and subtract it from every joint in every frame
#      of the whole sequence. This is pure translation. No rotation is
#      applied anywhere in the official pipeline.
#
#   2. Padding (align_frames): pad every sample to 300 frames (the maximum
#      sequence length in the dataset). For single-actor clips, the actor 2
#      slot is filled by DUPLICATING actor 1 data, not zero-filling it.
#
#   3. Splitting (get_indices / split_dataset): two official evaluation
#      protocols exist, and both are built here in a single pass:
#        - CSub (cross-subject): a fixed list of 53 official train subject
#          IDs out of 106 total. The remaining 53 subjects are test.
#        - CSet (cross-setup): even-numbered camera setup IDs (1-32) are
#          train, odd-numbered setup IDs are test.
#
# EARLIER BUG (fixed here, for CSub): an earlier version of this pipeline
# skipped step 1 entirely (no centering, raw camera coordinates), zero-padded
# single-actor clips instead of duplicating in step 2, and used an incorrect
# 55-subject train list in step 3 (incorrectly including subjects 96 and 99,
# which belong in test). Combined, these caused the pretrained CTR-GCN model
# to score 11.24 percent Top-1 instead of the expected 84.9 percent, because
# the model was trained on correctly centered and split data and received
# badly mismatched input.
#
# CSub was verified end to end: after fixing all three issues, CTR-GCN
# inference on the resulting test set scored 84.91 percent Top-1 and 97.33
# percent Top-5, matching the official published 84.9 percent almost exactly.
# CSet uses the identical centering and padding logic (only the split rule
# differs, taken directly from the official script) but has not yet been
# separately benchmarked through inference. Treat CSet numbers as
# structurally correct but not yet independently confirmed the way CSub is.
#
# DISK STRATEGY: rather than writing a scratch file and converting it to a
# proper .npy afterward (which briefly needs double the disk space), this
# cell writes directly into a properly-headered .npy file from the start,
# using numpy.lib.format.open_memmap. Every write already lands in its
# final, correctly-formatted location, so peak disk usage for both splits
# combined is only about one copy of the data (roughly 38 GB total), not two.
# ============================================================

import numpy as np
from numpy.lib.format import open_memmap
import pickle
import os
import time
import shutil

OUT_DIR_LOCAL = '/content/NTU120_build'
os.makedirs(OUT_DIR_LOCAL, exist_ok=True)

# --- Check local disk space before starting ---
# Full dataset size: 113945 samples x 300 frames x 150 values x 4 bytes
# is about 19.1 GB for one complete pass over all samples. CSub and CSet
# each need one such pass (train + test together), so combined final output
# is roughly 38 GB. RAM usage stays low throughout: only one sample
# (well under 1 MB) is held at a time during the loop, plus the input
# pickle (about 3 GB once loaded). A 12.7 GB Colab CPU session is enough.
total, used, free = shutil.disk_usage('/content')
print(f"Local disk free: {free/1024**3:.1f} GB (need at least 45 GB for both CSub and CSet)")
if free / 1024**3 < 45:
    raise RuntimeError("Not enough local disk space. Free up space before continuing.")

# --- Load per-sample metadata (same order as raw_denoised_joints.pkl) ---
performer = np.loadtxt(os.path.join(STAT_DIR, 'performer.txt'), dtype=int)
setup = np.loadtxt(os.path.join(STAT_DIR, 'setup.txt'), dtype=int)
label = np.loadtxt(os.path.join(STAT_DIR, 'label.txt'), dtype=int) - 1  # convert to 0-indexed

with open(LOCAL_DENOISED_PKL, 'rb') as f:
    skes_joints = pickle.load(f)  # list of 113945 arrays, each (num_frames, 75 or 150)

N = len(skes_joints)
assert N == 113945, f"Expected 113945 samples, got {N}"
print(f"Loaded {N} samples.")

# --- Official CSub train subject list (53 IDs) ---
# Read directly from the official seq_transformation.py source. Do not
# hand-derive this list; a previous attempt at doing so was wrong.
CSUB_TRAIN_IDS = [1, 2, 4, 5, 8, 9, 13, 14, 15, 16, 17, 18, 19, 25, 27, 28,
                   31, 34, 35, 38, 45, 46, 47, 49, 50, 52, 53, 54, 55, 56, 57,
                   58, 59, 70, 74, 78, 80, 81, 82, 83, 84, 85, 86, 89, 91, 92,
                   93, 94, 95, 97, 98, 100, 103]

csub_train_mask = np.isin(performer, CSUB_TRAIN_IDS)
csub_train_indices = np.where(csub_train_mask)[0]
csub_test_indices = np.where(~csub_train_mask)[0]
n_csub_train, n_csub_test = len(csub_train_indices), len(csub_test_indices)
print(f"CSub split -- train: {n_csub_train}, test: {n_csub_test} (expected 63026 / 50919)")
assert n_csub_train == 63026 and n_csub_test == 50919, "CSub split sizes do not match the official counts."

# --- Official CSet split: even setup IDs train, odd setup IDs test ---
cset_train_mask = (setup % 2 == 0)
cset_train_indices = np.where(cset_train_mask)[0]
cset_test_indices = np.where(~cset_train_mask)[0]
n_cset_train, n_cset_test = len(cset_train_indices), len(cset_test_indices)
print(f"CSet split -- train: {n_cset_train}, test: {n_cset_test}")

csub_train_pos = {idx: pos for pos, idx in enumerate(csub_train_indices)}
csub_test_pos = {idx: pos for pos, idx in enumerate(csub_test_indices)}
cset_train_pos = {idx: pos for pos, idx in enumerate(cset_train_indices)}
cset_test_pos = {idx: pos for pos, idx in enumerate(cset_test_indices)}

# --- Allocate output arrays directly as proper .npy files via open_memmap ---
# open_memmap writes a valid .npy header immediately, so unlike a plain
# np.memmap, these files can be loaded later with an ordinary np.load()
# with no separate conversion step required.
SAMPLE_SHAPE = (300, 150)

FINAL_CSUB_X_TRAIN = os.path.join(OUT_DIR_LOCAL, 'csub_x_train.npy')
FINAL_CSUB_X_TEST = os.path.join(OUT_DIR_LOCAL, 'csub_x_test.npy')
FINAL_CSET_X_TRAIN = os.path.join(OUT_DIR_LOCAL, 'cset_x_train.npy')
FINAL_CSET_X_TEST = os.path.join(OUT_DIR_LOCAL, 'cset_x_test.npy')

csub_x_train = open_memmap(FINAL_CSUB_X_TRAIN, mode='w+', dtype='float32', shape=(n_csub_train, *SAMPLE_SHAPE))
csub_x_test = open_memmap(FINAL_CSUB_X_TEST, mode='w+', dtype='float32', shape=(n_csub_test, *SAMPLE_SHAPE))
cset_x_train = open_memmap(FINAL_CSET_X_TRAIN, mode='w+', dtype='float32', shape=(n_cset_train, *SAMPLE_SHAPE))
cset_x_test = open_memmap(FINAL_CSET_X_TEST, mode='w+', dtype='float32', shape=(n_cset_test, *SAMPLE_SHAPE))

csub_y_train = np.zeros(n_csub_train, dtype=np.int64)
csub_y_test = np.zeros(n_csub_test, dtype=np.int64)
cset_y_train = np.zeros(n_cset_train, dtype=np.int64)
cset_y_test = np.zeros(n_cset_test, dtype=np.int64)

print("\nProcessing samples: centering, actor duplication, padding ...")
print("Each sample is written once and used by both CSub and CSet splits.")
t0 = time.time()

for i in range(N):
    ske = skes_joints[i].astype(np.float32).copy()  # (num_frames, 75 or 150)
    num_frames = ske.shape[0]
    num_bodies = 1 if ske.shape[1] == 75 else 2

    # --- Step 1: centering (seq_translation) ---
    # Find the first frame where actor 1 has real (nonzero) data.
    j = 0
    while j < num_frames and not np.any(ske[j, :75] != 0):
        j += 1
    if j == num_frames:
        j = 0  # fallback for the rare case of a fully empty actor 1

    origin = ske[j, 3:6].copy()  # spine-mid joint (joint index 1) at that frame

    if num_bodies == 1:
        ske -= np.tile(origin, 25)
    else:
        # Track frames that were originally all-zero for each actor, so we
        # can restore them to exactly zero after subtracting origin (otherwise
        # a genuinely missing frame would end up holding a fake -origin value).
        missing_1 = np.where(ske[:, :75].sum(axis=1) == 0)[0]
        missing_2 = np.where(ske[:, 75:].sum(axis=1) == 0)[0]
        ske -= np.tile(origin, 50)
        if len(missing_1) > 0:
            ske[missing_1, :75] = 0
        if len(missing_2) > 0:
            ske[missing_2, 75:] = 0

    # --- Step 2: padding to 300 frames (align_frames) ---
    # Single-actor clips duplicate actor 1 into actor 2's slot.
    padded = np.zeros(SAMPLE_SHAPE, dtype=np.float32)
    if num_bodies == 1:
        padded[:num_frames] = np.hstack((ske, ske))
    else:
        padded[:num_frames] = ske

    action_label = label[i]

    # --- Step 3: write this sample into both CSub and CSet outputs ---
    if i in csub_train_pos:
        pos = csub_train_pos[i]
        csub_x_train[pos] = padded
        csub_y_train[pos] = action_label
    else:
        pos = csub_test_pos[i]
        csub_x_test[pos] = padded
        csub_y_test[pos] = action_label

    if i in cset_train_pos:
        pos = cset_train_pos[i]
        cset_x_train[pos] = padded
        cset_y_train[pos] = action_label
    else:
        pos = cset_test_pos[i]
        cset_x_test[pos] = padded
        cset_y_test[pos] = action_label

    if (i + 1) % 20000 == 0:
        elapsed = time.time() - t0
        pct = (i + 1) / N * 100
        eta = (elapsed / (i + 1)) * (N - i - 1) / 60
        print(f"  {pct:.1f}% ({i+1}/{N}) -- {elapsed/60:.1f} min elapsed, ~{eta:.1f} min remaining")

for arr in [csub_x_train, csub_x_test, cset_x_train, cset_x_test]:
    arr.flush()
print(f"\nProcessing done in {(time.time()-t0)/60:.1f} min.")

# --- Save label arrays (small, plain np.save is fine here) ---
FINAL_CSUB_Y_TRAIN = os.path.join(OUT_DIR_LOCAL, 'csub_y_train.npy')
FINAL_CSUB_Y_TEST = os.path.join(OUT_DIR_LOCAL, 'csub_y_test.npy')
FINAL_CSET_Y_TRAIN = os.path.join(OUT_DIR_LOCAL, 'cset_y_train.npy')
FINAL_CSET_Y_TEST = os.path.join(OUT_DIR_LOCAL, 'cset_y_test.npy')

np.save(FINAL_CSUB_Y_TRAIN, csub_y_train)
np.save(FINAL_CSUB_Y_TEST, csub_y_test)
np.save(FINAL_CSET_Y_TRAIN, cset_y_train)
np.save(FINAL_CSET_Y_TEST, cset_y_test)

# Release the memmap file handles now that everything is written and flushed
del csub_x_train, csub_x_test, cset_x_train, cset_x_test

print("\n=== Cell 5 complete. Final .npy files ready locally (both CSub and CSet). ===")
for fname in ['csub_x_train.npy', 'csub_x_test.npy', 'csub_y_train.npy', 'csub_y_test.npy',
              'cset_x_train.npy', 'cset_x_test.npy', 'cset_y_train.npy', 'cset_y_test.npy']:
    fpath = os.path.join(OUT_DIR_LOCAL, fname)
    print(f"  {fname} -- {os.path.getsize(fpath)/1024**3:.3f} GB")


In [ ]:
# ============================================================
# PHASE 4.5 — CELL 6: Verify the corrected data, then copy to Drive
# Runtime: CPU only.
#
# Every check below must pass for a given split before that split is
# copied to Drive. CSub and CSet are checked and copied independently,
# so a problem with one does not block the other.
# ============================================================

import numpy as np
import os
import shutil
import time


def verify_split(name, x_train_path, x_test_path, y_train_path, y_test_path,
                  expected_train_shape, expected_test_shape):
    print("\n" + "=" * 60)
    print(f"Verifying {name}")
    print("=" * 60)

    x_train = np.load(x_train_path, mmap_mode='r')
    x_test = np.load(x_test_path, mmap_mode='r')
    y_train = np.load(y_train_path)
    y_test = np.load(y_test_path)

    ok = True

    print("--- Shape check ---")
    print(f"x_train: {x_train.shape}  x_test: {x_test.shape}")
    print(f"y_train: {y_train.shape}  y_test: {y_test.shape}")
    if x_train.shape != expected_train_shape or x_test.shape != expected_test_shape:
        print("FAIL: shape mismatch")
        ok = False

    print("\n--- Label range check ---")
    print(f"y_train: [{y_train.min()}, {y_train.max()}]  y_test: [{y_test.min()}, {y_test.max()}]")
    if y_train.min() < 0 or y_train.max() > 119 or y_test.min() < 0 or y_test.max() > 119:
        print("FAIL: label out of range")
        ok = False

    print("\n--- Empty sample check (2000 samples from each split) ---")
    rng = np.random.default_rng(0)
    n_check = min(2000, x_train.shape[0])
    sample_idx_tr = rng.choice(x_train.shape[0], n_check, replace=False)
    sample_idx_te = rng.choice(x_test.shape[0], min(2000, x_test.shape[0]), replace=False)
    empty_tr = sum(1 for i in sample_idx_tr if not np.any(x_train[i]))
    empty_te = sum(1 for i in sample_idx_te if not np.any(x_test[i]))
    print(f"Empty in train sample: {empty_tr}/{n_check}  Empty in test sample: {empty_te}/{len(sample_idx_te)}")
    if empty_tr > 0 or empty_te > 0:
        print("FAIL: found fully-empty samples")
        ok = False

    print("\n--- Centering check (5 samples) ---")
    check_indices = [i for i in [0, 100, 5000, 20000, 40000] if i < x_train.shape[0]]
    for i in check_indices:
        sample = x_train[i]
        nonzero_frames = np.where(np.any(sample[:, :75] != 0, axis=1))[0]
        if len(nonzero_frames) == 0:
            continue
        first_frame = nonzero_frames[0]
        joint1 = sample[first_frame, 3:6]
        print(f"  sample {i}: first_frame={first_frame}, spine-mid joint = {joint1}")
        if not np.allclose(joint1, 0, atol=1e-4):
            print("    WARNING: expected approximately [0, 0, 0]")
            ok = False

    print("\n--- Single-actor duplication check (10 samples) ---")
    dup_found = 0
    two_actor_found = 0
    for i in range(min(10, x_train.shape[0])):
        sample = x_train[i]
        nonzero_frames = np.where(np.any(sample != 0, axis=1))[0]
        if len(nonzero_frames) == 0:
            continue
        f = nonzero_frames[0]
        actor1, actor2 = sample[f, :75], sample[f, 75:]
        if np.allclose(actor1, actor2):
            dup_found += 1
        elif np.any(actor2 != 0):
            two_actor_found += 1
    print(f"Duplicated single-actor samples: {dup_found}, distinct two-actor samples: {two_actor_found} (out of first 10)")

    status_text = "ALL CHECKS PASSED" if ok else "CHECKS FAILED"
    print(f"\n{name}: {status_text}")
    return ok


def copy_split_to_drive(name, drive_out_dir, x_train_path, x_test_path, y_train_path, y_test_path):
    os.makedirs(drive_out_dir, exist_ok=True)
    files_to_copy = [
        (x_train_path, os.path.join(drive_out_dir, 'x_train.npy')),
        (x_test_path, os.path.join(drive_out_dir, 'x_test.npy')),
        (y_train_path, os.path.join(drive_out_dir, 'y_train.npy')),
        (y_test_path, os.path.join(drive_out_dir, 'y_test.npy')),
    ]
    print(f"\n--- Copying {name} to Drive: {drive_out_dir} ---")
    for src, dst in files_to_copy:
        src_size = os.path.getsize(src)
        print(f"Copying {os.path.basename(dst)} ({src_size/1024**3:.3f} GB) ...", end=" ")
        t0 = time.time()
        shutil.copy2(src, dst)
        dst_size = os.path.getsize(dst)
        status = "OK" if dst_size == src_size else "SIZE MISMATCH"
        print(f"{status} ({(time.time()-t0)/60:.1f} min)")
        if status != "OK":
            raise RuntimeError(f"Copy failed for {src}. Re-run this cell.")

    # Confirm the files copied to Drive can actually be loaded back correctly.
    # This exact failure mode (a memmap file with no .npy header) has
    # happened before with an earlier version of this pipeline, so it is
    # worth confirming directly rather than assuming the copy is correct.
    print(f"Verifying {name} files load correctly directly from Drive ...")
    check_x_test = np.load(os.path.join(drive_out_dir, 'x_test.npy'), mmap_mode='r')
    print(f"Loaded from Drive OK. x_test shape: {check_x_test.shape}, dtype: {check_x_test.dtype}")
    del check_x_test


# --- CSub: this is the split verified through actual CTR-GCN inference ---
csub_ok = verify_split(
    "NTU120 CSub",
    FINAL_CSUB_X_TRAIN, FINAL_CSUB_X_TEST, FINAL_CSUB_Y_TRAIN, FINAL_CSUB_Y_TEST,
    expected_train_shape=(63026, 300, 150), expected_test_shape=(50919, 300, 150),
)
if csub_ok:
    copy_split_to_drive(
        "NTU120 CSub", DRIVE_CSUB_OUT,
        FINAL_CSUB_X_TRAIN, FINAL_CSUB_X_TEST, FINAL_CSUB_Y_TRAIN, FINAL_CSUB_Y_TEST,
    )
    print("\nCSub ready for CTR-GCN inference in the Phase 5 notebook.")
    print("Expected result: Top-1 approximately 84.9 percent, Top-5 approximately 97.3 percent.")
else:
    print("\nCSub checks failed. Not copied to Drive. Review the output above.")

# --- CSet: same method, structurally checked, not yet benchmarked through inference ---
cset_ok = verify_split(
    "NTU120 CSet",
    FINAL_CSET_X_TRAIN, FINAL_CSET_X_TEST, FINAL_CSET_Y_TRAIN, FINAL_CSET_Y_TEST,
    expected_train_shape=(n_cset_train, 300, 150), expected_test_shape=(n_cset_test, 300, 150),
)
if cset_ok:
    copy_split_to_drive(
        "NTU120 CSet", DRIVE_CSET_OUT,
        FINAL_CSET_X_TRAIN, FINAL_CSET_X_TEST, FINAL_CSET_Y_TRAIN, FINAL_CSET_Y_TEST,
    )
    print("\nCSet copied to Drive. Structurally verified (shapes, labels, centering,")
    print("duplication all correct) but not yet confirmed through an actual CTR-GCN")
    print("inference run the way CSub was. Run inference before trusting the accuracy number.")
else:
    print("\nCSet checks failed. Not copied to Drive. Review the output above.")

print("\n=== Cell 6 complete. ===")
